# Patient transfer adaptation through a crisis: The dynamics of a Hospitals Network during the COVID-19 pandemic.

**Autores:** Cicchini Tomás, Salgado Ariel, Otero Lisandro, Goldman Mariano, Yacobitti Alejandro, Doldan Victoria, Kochen Silvia, Boechi Leonardo, Caridi Inés.

_Instituto del Cálculo (UBA-CONICET), Hospital de Alta Complejidad en Red Nestor Carlos Kirchner, ENYS (CONICET-HEC-UNAJ), y desarrollador free-lance._


In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))
from src.config import *
from src.io import *
from src.procesamiento import *
from src.visualizacion import *
from src.funciones_complejas import *

import sys
sys.path.append("..")  # agrega raiz del proyecto al path

# from scripts import bases, init_notebook as init

from src.config import crear_directorios_overleaf
crear_directorios_overleaf()

PermissionError: [WinError 32] El proceso no tiene acceso al archivo porque está siendo utilizado por otro proceso: 'graficos_overleaf\\1_general\\gen_mapa_redsudeste_global.pdf'

In [ ]:
# Carga todos los datos del proyecto de una vez
ctx_datos = init_notebook(data_path="../data")

df_pacientes    = ctx_datos["df_pacientes"]
traslados       = ctx_datos["traslados"]
hosp_coords     = ctx_datos["hosp_coords"]
municipios      = ctx_datos["municipios"]
municipios_amba = ctx_datos["municipios_amba"]


## The hospital network and the data.

**Contexto:** El trabajo en red permite optimizar recursos en momentos críticos.  
**Impacto COVID-19:** La pandemia puso en crisis a los sistemas de salud y obligó a adaptar infraestructuras.  
**Red Sudeste:** El sistema informático recolectó datos de camas y traslados de 14 hospitales de la Red Sudeste. Abarca cuatro municipios: Quilmes (QU), Almirante Brown (AB), Florencio Varela (FV) y Berazategui (BE), cubriendo 661 km² y casi 2 millones de habitantes.


In [ ]:
# Mapas (Paneles A, B y C)
# A. Aquí se visualiza la Provincia de Buenos Aires (PBA)

# ----------------------------
# cargar datos
# ----------------------------

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import matplotlib.lines as mlines
import ast
import os # <-- IMPORTANTE: Agregar esto para manejar las carpetas

# -------------------------
# 1. Cargar datos
# -------------------------
paises = gpd.read_file("../data/ne_countries/ne_110m_admin_0_countries.shp")
sudamerica = paises[paises["CONTINENT"] == "South America"]

provincias = gpd.read_file("../data/provincias_ign/provinciaPolygon.shp")
pba = provincias[provincias["nam"] == "Buenos Aires"]

# -------------------------
# 2. PROYECCIÓN (El truco para que no se vea estirado)
# -------------------------
sudamerica = sudamerica.to_crs(epsg=3857)
pba = pba.to_crs(epsg=3857)

# -------------------------
# 3. Plot con mejoras estéticas
# -------------------------
# Mantenemos el dpi=80 acá para la visualización en pantalla, 
# la calidad de exportación la definiremos en el savefig.
fig, ax = plt.subplots(figsize=(6, 8), dpi=80)

# Sudamérica (Fondo)
sudamerica.plot(
    ax=ax,
    color="white",      
    edgecolor="black",  
    linewidth=0.5        
)

# Buenos Aires resaltada
pba.plot(
    ax=ax,
    color="#808080",      
    edgecolor="black",
    linewidth=0.4            
)

# -------------------------
# 4. Ajustes de encuadre
# -------------------------
minx, miny, maxx, maxy = sudamerica.total_bounds
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)

# Título con mejor fuente
# ax.set_title(
#     "Ubicación de Buenos Aires en Sudamérica",
#     fontsize=12,
#     fontweight="bold",
#     pad=20,
#     loc="center"
# )

ax.axis("off")

plt.tight_layout()

# -------------------------
# 5. Exportación de Alta Calidad
# -------------------------
# Definir la ruta y crearla si no existe
ruta_salida = "results/outputs/geo"
os.makedirs(ruta_salida, exist_ok=True)

nombre_archivo = f"{ruta_salida}/mapa_buenos_aires"

# Exportar como PNG (Raster)
# dpi=300 es estándar de impresión. Puedes subirlo a 600 si necesitas extremo detalle.
plt.savefig(
    f"{nombre_archivo}.png", 
    dpi=300, 
    bbox_inches="tight", # Asegura fondo blanco (a veces los notebooks lo hacen transparente)
    transparent=True
)

# Exportar como Vectorial (SVG y PDF)
# SVG es ideal para editar en Illustrator/Inkscape o usar en web.
plt.savefig(
    f"{nombre_archivo}.svg", 
    format="svg", 
    bbox_inches="tight",
    transparent=True
)

# PDF es el formato preferido si vas a insertarlo en un documento de LaTeX o Word sin perder calidad.
guardar_pdf('gen_mapa_buenos_aires_global', subcarpeta='general')

# Siempre mostrar el gráfico DESPUÉS de guardarlo. 
# Si haces show() antes, savefig() guardará una imagen en blanco.
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import os # Asegurarnos de importar os

# -------------------------
# Cargar municipios (partidos)
# -------------------------
deptos = cargar_municipios("../data/shapefiles/departamento/departamentoPolygon.shp")

# Solo partidos de la provincia de Buenos Aires
pba = deptos[
    deptos["in1"].astype(str).str.startswith("06")
]

# -------------------------
# Partidos de la red sudeste
# -------------------------
red_sudeste = [
    "QUILMES", "ALMIRANTE BROWN", "FLORENCIO VARELA",
    "BERAZATEGUI", "LANUS", "LOMAS DE ZAMORA",
    "AVELLANEDA", "MORON", "ITUZAINGO"
]

sudeste = pba[pba["nam_limpio"].isin(red_sudeste)]

# Definir y crear la ruta base de salida si no existe
ruta_salida = "results/outputs/geo"
os.makedirs(ruta_salida, exist_ok=True)


# =====================================================
# OPTION 1
# Todos los partidos + red sudeste resaltada
# =====================================================

fig, ax = plt.subplots(figsize=(6,6))

# Todos los partidos
pba.plot(
    ax=ax,
    color="lightgray",
    edgecolor="black",
    linewidth=0.2
)

# Partidos de la red sudeste
sudeste.plot(
    ax=ax,
    color="dimgray",
    edgecolor="black",
    linewidth=0.3
)

# ax.set_title(
#     "Ubicación de la Red Sudeste en la PBA",
#     fontsize=13,
#     pad=10
# )

ax.axis("off")

plt.tight_layout()

# --- Exportación Opción 1 ---
nombre_archivo_1 = f"{ruta_salida}/mapa_red_sudeste_opcion1"

plt.savefig(f"{nombre_archivo_1}.png", dpi=300, bbox_inches="tight", transparent=True)
plt.savefig(f"{nombre_archivo_1}.svg", format="svg", bbox_inches="tight", transparent=True)
guardar_pdf('gen_mapa_redsudeste_opt1_global', subcarpeta='general')

plt.show() # Mostrar el gráfico de la opción 1


# =====================================================
# OPTION 2
# Provincia blanca sin divisiones + red sudeste
# =====================================================

# Crear geometría de toda la provincia
pba_union = pba.dissolve()

fig, ax = plt.subplots(figsize=(6,6))

# Provincia completa
pba_union.plot(
    ax=ax,
    color="white",
    edgecolor="black",
    linewidth=0.3
)

# Partidos de la red
sudeste.plot(
    ax=ax,
    color="dimgray",
    edgecolor="none"
)

# ax.set_title(
#     "Ubicación de la Red Sudeste en la PBA",
#     fontsize=13,
#     pad=10
# )

ax.axis("off")

plt.tight_layout()

# --- Exportación Opción 2 ---
nombre_archivo_2 = f"{ruta_salida}/mapa_red_sudeste_opcion2"

plt.savefig(f"{nombre_archivo_2}.png", dpi=300, bbox_inches="tight", transparent=True)
plt.savefig(f"{nombre_archivo_2}.svg", format="svg", bbox_inches="tight", transparent=True)
guardar_pdf('gen_mapa_redsudeste_opt2_global', subcarpeta='general')

plt.show() # Mostrar el gráfico de la opción 2

In [ ]:
# C. Los asentamientos cercanos

# RE.NA.BA.P

In [ ]:

# ==============================================================================
# CONFIGURACIÓN Y CARGA DE DATOS
# ==============================================================================
# Rutas de los archivos proporcionadas por el usuario. 
# Asegúrate de que los archivos existan en estas ubicaciones.
print("Cargando datos...")
deptos = gpd.read_file(RUTA_MUNICIPIOS)
barrios = gpd.read_file(RUTA_BARRIOS_POPULARES)

# Filtrar solo la PBA (códigos que empiezan con 06)
pba = deptos[deptos["in1"].astype(str).str.startswith("06")].copy()

# ==============================================================================
# 2. LIMPIEZA Y FILTRADO (LA SOLUCIÓN AL ERROR)
# ==============================================================================
# Creamos una columna limpia: todo a mayúsculas y sin tildes
pba["nam_limpio"] = (
    pba["nam"]
    .astype(str)
    .str.upper()
    .str.replace('Á', 'A')
    .str.replace('É', 'E')
    .str.replace('Í', 'I')
    .str.replace('Ó', 'O')
    .str.replace('Ú', 'U')
)

red_sudeste_names = [
    "QUILMES", "ALMIRANTE BROWN", "FLORENCIO VARELA",
    "BERAZATEGUI", "LANUS", "LOMAS DE ZAMORA",
    "AVELLANEDA", "MORON", "ITUZAINGO"
]

# Ahora sí filtramos de forma segura
sudeste = pba[pba["nam_limpio"].isin(red_sudeste_names)]

# Control de seguridad para evitar el error de NaN
if sudeste.empty:
    raise ValueError("¡Atención! El filtro sigue sin encontrar municipios. Revisa la columna 'nam' de tu shapefile.")

# ==============================================================================
# 3. PROYECCIÓN Y CRUCE ESPACIAL
# ==============================================================================
print("Proyectando a Web Mercator...")
pba = pba.to_crs(epsg=3857)
sudeste = sudeste.to_crs(epsg=3857)
barrios = barrios.to_crs(epsg=3857)

# Solo nos quedamos con los barrios que caen dentro de nuestros municipios
print("Cruzando capas...")
barrios_enfoque = gpd.clip(barrios, sudeste) 
# Nota: clip es más preciso visualmente que sjoin si un barrio cruza el límite

# ==============================================================================
# 4. GRAFICACIÓN EXACTA A LA IMAGEN
# ==============================================================================
print("Generando mapa...")
fig, ax = plt.subplots(figsize=(8, 8), dpi=80)

# A. Municipios de fondo (Gris clarito)
pba.plot(
    ax=ax,
    color="none",
    edgecolor="#B0B0B0", # Gris intermedio para los límites de fondo
    linewidth=0.8,
    zorder=1
)

# B. Municipios de la Red Sudeste (Borde negro grueso)
sudeste.plot(
    ax=ax,
    color="none",
    edgecolor="black",
    linewidth=1.5,      # Línea más gruesa para destacar
    zorder=2
)

# C. Barrios Populares (Rojo puro)
barrios_enfoque.plot(
    ax=ax,
    color="red",
    edgecolor="none",
    zorder=3
)

# ==============================================================================
# 5. AJUSTES FINALES (ZOOM Y TEXTO)
# ==============================================================================
ax.axis("off")

# Hacemos zoom en la zona de la red sudeste
minx, miny, maxx, maxy = sudeste.total_bounds
margen = 0.05 * (maxx - minx) # 5% de margen
ax.set_xlim(minx - margen, maxx + margen)
ax.set_ylim(miny - margen, maxy + margen)

# Agregamos la letra [C] arriba a la derecha (estilo igual a tu imagen)
ax.text(
    0.95, 0.95, 
    "[C]", 
    transform=ax.transAxes, 
    fontsize=45,           # Bien grande
    fontweight="bold", 
    color="black",         # Letra negra
    ha="right", 
    va="top",
    zorder=4
)

# El marco negro alrededor de toda la imagen
fig.patch.set_linewidth(3)
fig.patch.set_edgecolor('black')

plt.tight_layout()
guardar_pdf('gen_mapa_barrios_populares_global', subcarpeta='general')
plt.show()